# Imports

In [1]:
import sys
sys.path.append('..')
from tqdm.auto import tqdm
from src import helpers

/home/abdelnour-main/miniconda3/envs/text2kg-main-2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load data

In [2]:
path = "/home/abdelnour-main/Documents/Projects/Text2KG/old_data/expertise_maps/nantes-uni-2020--Conv_Attn_null_False/dump.json"
raw = helpers.load_json_file(path)

In [3]:
### id <-> Expertise
id2Expertise = raw['nodes']['Expertise']
id2Expertise = dict(zip([e['id'] for e in id2Expertise], id2Expertise))

### id <-> Entity
id2Entity = raw['nodes']['Entity']
id2Entity = dict(zip([e['id'] for e in id2Entity], id2Entity))

### Join
for e in tqdm(raw['edges']['Entity_BelongsTo_Expertise']):

    en = id2Entity[e['src']]
    ex = id2Expertise[e['dst']]

    if 'entities' not in ex:
        ex['entities'] = []

    ex['entities'].append(en)

expertises = [{**ex,'size' : len(ex['entities'])} for ex in id2Expertise.values()]

100%|██████████| 38273/38273 [00:00<00:00, 163534.36it/s]


In [4]:
def process_exterise(
    expertise : dict,
    k : int = 30,
    sort_attr : str = 'specificity_weighted_pagerank'
) -> dict | None:
    
    entities = sorted(
        expertise['entities'], 
        key=lambda x : x[sort_attr], 
        reverse=True
    )

    expertise['entities'] = entities[:min(k, len(entities))]

    if len(expertise['entities']) <= 15:
        return None


    return expertise

In [5]:
expertises = list(filter(None, map(process_exterise, expertises)))

In [33]:
import asyncio
from openai import AsyncClient
from random import shuffle

client = AsyncClient(
    api_key="sk-46e0b8e6cb7d4c989b13af3980c397c3",
    base_url="https://api.deepseek.com"
)

async def evaluate_community(client: AsyncClient, entities: list[str], k: int = 5) -> list[str]:

    system_prompt = (
        "You are an expert evaluator of scientific and technical domains.\n"
        "You will receive a list of entities representing a knowledge cluster.\n\n"
        "Your task:\n"
        "For each entity, assign a relevance score from 0 to 10 indicating how well it represents the core expertise of the cluster.\n"
        "Ignore order. Evaluate each entity independently.\n\n"
        "Return ONLY a pipe-separated list in the format:\n"
        "entity|score\n"
        "one per line, no explanations."
    )

    user_prompt = "\n".join(entities)

    response = await client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        top_p=1,
        seed=42,
    )

    content = response.choices[0].message.content

    scored = []
    for line in content.strip().split("\n"):
        if "|" in line:
            ent, score = line.split("|")
            scored.append((ent.strip(), float(score.strip())))

    # deterministic ranking in code (important!)
    scored.sort(key=lambda x: x[1], reverse=True)

    return [e for e, _ in scored[:k]]

async def evaluate(
    client : AsyncClient, 
    communities : list[dict], 
    k : int = 5,
    batch_size : int = 5
) -> tuple[dict, list[dict]]:
    
    batch = []
    all_results = []

    metrics = {
        'top-1' : 0.0,
        'top-3' : 0.0,
        'top-5' : 0.0,
    }

    for community in communities:

        name = community['name']
        entities = [e['name'] for e in community['entities']]
        shuffle(entities)

        batch.append((name, entities))

        if len(batch) == batch_size:

            results = await asyncio.gather(*[
                evaluate_community(
                    client, 
                    entities, 
                    k
                )
                for label, entities in batch
            ])

            all_results.extend([
                {
                    "name" : name, # .replace(' ','_'),
                    f'top-{k}' : top_k,
                    "entities" : entities,
                }
                for (name, entities), top_k in zip(batch, results)
            ])

            batch = []

    if len(batch) > 0:

        results = await asyncio.gather(*[
            evaluate_community(
                client, 
                entities, 
                k
            )
            for label, entities in batch
        ])

        all_results.extend([
            {
                "name" : name , # .replace(' ','_'),
                f'top-{k}' : top_k,
                "entities" : entities
            }
            for (name, entities), top_k in zip(batch, results)
        ])

        batch = []

    for result in all_results:
        metrics['top-1'] += int(result['name'] == result[f'top-{k}'][0]) / len(all_results)
        metrics['top-3'] += int(result['name'] in result[f'top-{k}'][:3]) / len(all_results)
        metrics['top-5'] += int(result['name'] in result[f'top-{k}'][:5]) / len(all_results)

    return metrics, all_results

In [34]:
metrics, all_results = await evaluate(
    client,
    expertises,
    k=10,
    batch_size=5
)

In [35]:
metrics

{'top-1': 0.1030042918454934,
 'top-3': 0.20171673819742522,
 'top-5': 0.2861230329041497}

In [37]:
all_results

[{'name': 'workspace',
  'top-10': ['parallel manipulator',
   'parallel robots',
   'positive cable tensions',
   'workspace',
   'wrench closure workspace',
   'cable cable collisions',
   'cable dynamics',
   'large translational workspace',
   'high cable tension',
   'cdpr transparency'],
  'entities': ['replicas',
   'cable loop',
   'robot current state',
   'parallel manipulator',
   'translation',
   'metal plates',
   'cable cable collisions',
   'unwanted torques',
   'cuspidality property',
   'five bar mechanism',
   'positive cable tensions',
   'task specification',
   'elastic springs',
   'cable dynamics',
   'prototype structural stability',
   'internal swelling reaction',
   'heavy loads',
   'point mass end effector',
   'large translational workspace',
   'rectangular workspace',
   'high cable tension',
   'detection performance',
   'parallel signal processing applications',
   'workspace',
   'average size',
   'virtual point',
   'rotation',
   'wrench closure

In [1]:
import pandas as pd

df = pd.read_json('/home/abdelnour-main/Documents/Projects/Text2KG/data/expertise_maps/nantes_uni_2020_2026_e5-v2/communities.json')

In [2]:
df.head().transpose()

,0,1,2,3,4
id,0,1,2,3,4
entities,"[{'id': 'controllability', 'name': 'controllab...","[{'id': 'adhesive_strength', 'name': 'adhesive...","[{'id': 'data_center', 'name': 'data center', ...","[{'id': 'gas_evolution_profile', 'name': 'gas ...","[{'id': 'global_warming_potential', 'name': 'g..."
labels,"[[robotics, 0.9999971389770501]]","[[eco-responsible_processes, 0.503706395626068...","[[eco-responsible_processes, 0.8697872757911681]]","[[monitoring_and_analysis, 0.9768187999725341]]","[[eco-design, 0.9905457496643061], [eco-respon..."
labs,"[[LS2N, 0.9999995231628411]]","[[GEM, 0.999821722507476]]",[],[],"[[GEPEA, 0.8625637888908381]]"
specificity_weighted_pagerank_name,workspace,cement paste,data center,heat treatment,carbon footprint
personalized_pagerank_name,point mass payload,thermal curing,data center,daylight hour,sugar
llm_label,Kinematic design and control of serial-paralle...,Geopolymer-based cementitious systems with chl...,Data center energy optimization under climatic...,Microwave-assisted fluidized bed processing,Sustainable food-energy-water nexus analysis


In [3]:
df['entities'] = df['entities'].apply(lambda x : [e['name'] for e in x])
df['domains'] = df['labels'].apply(lambda x : [e[0] for e in x])
df['labs'] = df['labs'].apply(lambda x : [e[0] for e in x])

df = df.drop(columns=['labels', 'specificity_weighted_pagerank_name'])

In [23]:
subset = df[df['entities'].apply(lambda x : len(x) <= 30)]
subset = subset.sample(100)

In [28]:
domains_counts = {}

for domains_list in subset['domains'].tolist():
    for domain in domains_list:
        domains_counts[domain] = domains_counts.get(domain, 0) + 1

In [29]:
domains_counts

{'industrial_digitalization': 33,
 'composites_and_new_materials': 39,
 'monitoring_and_analysis': 31,
 'digital_engineering_of_products_and_processes': 16,
 'robotics': 6,
 'eco-responsible_processes': 6,
 'processes_and_assembly': 6,
 'additive_manufacturing': 3,
 'decarbonization_of_mobility': 2,
 'industrial_organization_performance': 7,
 'augmentation_of_human_capabilities_in_manufacturing': 5,
 'decarbonization_of_energy': 7,
 'recycling_/_reuse': 1,
 'reduction_and_treatment_of_effluents_and_waste': 3,
 'user-centered_design': 3}

In [31]:
subset['domains'] = subset['domains'].apply('\n'.join)
subset['labs'] = subset['labs'].apply('\n'.join)
subset['entities'] = subset['entities'].apply('\n'.join)

In [32]:
subset.to_csv('subset.csv')